In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train = torch.tensor(X_train, dtype=torch.float32)  # <Replace None with your code>
X_test  = torch.tensor(X_test, dtype=torch.float32)  # <Replace None with your code>
y_train = torch.tensor(y_train, dtype=torch.float32)  # <Replace None with your code> !!!!
y_test  = torch.tensor(y_test, dtype=torch.float32)  # <Replace None with your code>!!!!!!


In [ ]:
# 2. Create TensorDataset objects


train_dataset = TensorDataset(X_train, y_train)  # <Replace None with your code>
test_dataset = TensorDataset(X_test, y_test)   # <Replace None with your code>

In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) #Create a DataLoader that loads training data in random batches of 32 samples.
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # We do NOT shuffle test data




In [ ]:
# 4. Print shape of one batch

# Print dataset sizes
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}") # [batch_size, number_of_features]
print(f"Training batch labels shape: {y_batch.shape}") # One label per sample

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

sample_index = 4
sample_image, sample_label = train_dataset[sample_index]  # Unpack the tuple

sample_image = sample_image.permute(1, 2, 0).numpy()  # Change shape to (H, W, C) for display
sample_label = sample_label.item()

plt.imshow(sample_image)
plt.title(f'Sample Image Label: {sample_label}')
plt.axis('off')
plt.show()



In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim): # !!!!!
    super(NN4Layer, self).__init__()
    # TODO: Define the first linear layer: input_dim -> hidden_dim
    self.layer1 = nn.Linear(input_dim, hidden_dim)  # <Replace None with your code>

    # TODO: Define the second linear layer: hidden_dim -> hidden_dim
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)  # <Replace None with your code>

    # TODO: Define the output layer: hidden_dim -> 1 (single value for regression) !!!!
    self.layer3 = nn.Linear(hidden_dim,hidden_dim)  # Output layer → ONE value for regression
    self.layer4 = nn.Linear(hidden_dim,1)  # Output layer → ONE value for regression



    # TODO: Define ReLU activation
    self.relu = nn.ReLU()  # <Replace None with your code>

  def forward(self, x):
    # TODO: First hidden layer with ReLU
    a1 = self.relu(self.layer1(x))  # <Replace None with your code>

    # TODO: Second hidden layer with ReLU
    a2 = self.relu(self.layer2(a1))  # <Replace None with your code>
    a3 = self.relu(self.layer3(a2))  # <Replace None with your code>

    # TODO: Output layer (no activation for regression)
    output = self.layer4(a3)   # <Replace None with your code>

    return output

In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # TODO: Set the model to training mode
  model.train()

  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # TODO: Move batch to the selected device
    X_batch = X_batch.to(device)   # <Replace None with your code>
    y_batch = y_batch.view(-1, 1).to(device)   # <Replace None with your code> [HINT: reshape to (-1, 1)] !!!

    # TODO: Forward pass - get model predictions
    outputs = model(X_batch)  # <Replace None with your code>

    # TODO: Compute loss using criterion
    loss = criterion(outputs, y_batch)  # <Replace None with your code>

    # TODO: Backward pass & optimization
    # Step 1: Clear previous gradients
    optimizer.zero_grad()

    # Step 2: Compute gradients (backward pass)
    loss.backward()

    # Step 3: Update model parameters
    optimizer.step()

    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, criterion, test_loader, device):
  # TODO: Set the model to evaluation mode
  model.eval()

  running_loss = 0.0  # sum of batch losses
  correct = 0  # number of correct predictions
  total = 0  # total number of samples


  # TODO: Disable gradient computation using torch.no_grad()
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      # TODO: Move data to device
      X_batch = X_batch.to(device)     # <Replace None with your code>
      y_batch = y_batch.float().view(-1, 1).to(device)    # <Replace None with your code> [HINT: reshape to (-1, 1)] !!!!! # shape: (batch_size, 1)

      # TODO: Forward pass - get model predictions
      outputs = model(X_batch)  # # shape: (batch_size, 1)

      # TODO: Compute loss using criterion
      loss = criterion(outputs, y_batch)  # <Replace None with your code>

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
# TODO: Set up the device (use GPU if available, otherwise CPU)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = X_train.shape[1]   # Number of  features
hidden_dim = 64                # Design choice (feel free to experiment!)

# TODO: Instantiate the model and move it to the device
model = NN4Layer(input_dim, hidden_dim).to(device)  # <Replace None with your code>

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

# Hyperparameters (feel free to experiment!)
num_epochs = 20
learning_rate = 0.001

# TODO: Define criterion (loss function) - use MSELoss for regression
criterion = nn.MSELoss()  # <Replace None with your code>

# TODO: Define optimizer - use AdamW with the model parameters and learning rate
optimizer = AdamW(model.parameters(), learning_rate)  # <Replace None with your code>

In [ ]:
# Task 5: Start training for 20 epochs:

# Run Training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # TODO: Train one epoch using the train_one_epoch function
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)  # <Replace None with your code>

  # TODO: Validate using the validate function
  val_loss = validate(model, criterion, test_loader, device)  # <Replace None with your code>

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:

# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: Plz stage 4 :(, I LOVE ANON